# RAG Chatbot — Blockchain Whitepapers
**NoLimit Indonesia — Data Scientist Intern Technical Test**

**Task:** Option C — Chatbot with Retrieval-Augmented Generation (RAG)

**Documents:** Bitcoin, Ethereum, and Solana whitepapers (PDF)

**Pipeline Overview:**
```
PDF Upload → Text Extraction → Chunking → Embedding (MiniLM) → FAISS Index
                                                                      ↓
User Query → Query Embedding → Similarity Search → Top-K Chunks → Prompt → LLM → Answer + Citation
```

## Install Dependencies

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.7 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF
import faiss
import numpy as np
import os
import textwrap
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

## Upload PDFs & Extract Text


In [ ]:
from google.colab import files

# ── UPLOAD FILES HERE
uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))

Saving bitcoin.pdf to bitcoin (1).pdf
Saving etherium.pdf to etherium (1).pdf
Saving solana.pdf to solana (1).pdf
Uploaded files: ['bitcoin (1).pdf', 'etherium (1).pdf', 'solana (1).pdf']


In [ ]:
def extract_text_from_pdf(file_path):
    """Extract text per page from a PDF, returning list of (page_num, text)."""
    doc = fitz.open(file_path)
    pages = []
    for page_num in range(len(doc)):
        text = doc[page_num].get_text("text")  # extract plain text, skip images
        text = text.strip()
        if text:  # skip blank pages
            pages.append((page_num + 1, text))
    doc.close()
    return pages


# Extract text from all uploaded PDFs
raw_documents = {}
for filename in uploaded.keys():
    raw_documents[filename] = extract_text_from_pdf(filename)
    print(f"{filename}: {len(raw_documents[filename])} pages extracted")

bitcoin (1).pdf: 9 pages extracted
etherium (1).pdf: 36 pages extracted
solana (1).pdf: 32 pages extracted


## Chunking with Metadata

Each page's text is split into overlapping chunks to preserve context at boundaries. Every chunk stores `file_name` and `page_number` for citation.

In [ ]:
CHUNK_SIZE    = 1000  # characters per chunk
CHUNK_OVERLAP = 150   # overlap between consecutive chunks


def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping character-level chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap  # slide window with overlap
    return chunks


# Build a flat list of chunk dicts with metadata
all_chunks = []
for file_name, pages in raw_documents.items():
    for page_num, page_text in pages:
        for chunk in chunk_text(page_text):
            all_chunks.append({
                "text":        chunk,
                "file_name":   file_name,
                "page_number": page_num
            })

print(f"Total chunks created: {len(all_chunks)}")
print("\nSample chunk:")
print(all_chunks[0])

Total chunks created: 248

Sample chunk:
{'text': "Bitcoin: A Peer-to-Peer Electronic Cash System\nSatoshi Nakamoto\nsatoshin@gmx.com\nwww.bitcoin.org\nAbstract.  A purely peer-to-peer version of electronic cash would allow online \npayments to be sent directly from one party to another without going through a \nfinancial institution.  Digital signatures provide part of the solution, but the main \nbenefits are lost if a trusted third party is still required to prevent double-spending. \nWe propose a solution to the double-spending problem using a peer-to-peer network. \nThe network timestamps transactions by hashing them into an ongoing chain of \nhash-based proof-of-work, forming a record that cannot be changed without redoing \nthe proof-of-work.  The longest chain not only serves as proof of the sequence of \nevents witnessed, but proof that it came from the largest pool of CPU power.  As \nlong as a majority of CPU power is controlled by nodes that are not cooperating to \nattack 

## Embedding & FAISS Index

Using `all-MiniLM-L6-v2` — a fast, lightweight sentence embedding model from Hugging Face.

In [ ]:
# Load embedding model (runs locally, no API needed)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encode all chunks into dense vectors
print("Encoding chunks... this may take a minute.")
chunk_texts = [c["text"] for c in all_chunks]
embeddings  = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

# Build a flat L2 FAISS index
dimension   = embeddings.shape[1]              # vector size (384 for MiniLM)
index       = faiss.IndexFlatL2(dimension)     # exact nearest-neighbour search
index.add(embeddings.astype(np.float32))       # add all vectors to the index

print(f"FAISS index built — {index.ntotal} vectors, dimension {dimension}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding chunks... this may take a minute.


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

FAISS index built — 248 vectors, dimension 384


## LLM Setup

Using **Qwen2.5-0.5B-Instruct** — the lightest instruction-tuned model in the Qwen family (~1 GB), free on Hugging Face, and well-suited for RAM-constrained Colab environments.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # half-precision to halve VRAM usage
    device_map="auto"            # auto-place on GPU if available, else CPU
)

# Wrap into HF pipeline for easy inference
llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,             # greedy decoding — deterministic, faster
)

print("LLM ready:", MODEL_ID)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready: Qwen/Qwen2.5-0.5B-Instruct


## RAG Pipeline

The retriever fetches the top-K most relevant chunks via FAISS, injects them into a prompt, and the LLM generates an answer. The prompt is intentionally minimal — it tells the model to answer only from the given context and adapt its language to the user naturally.

In [ ]:
TOP_K = 4  # number of chunks to retrieve per query


def retrieve(query, top_k=TOP_K):
    """Embed query and retrieve top-k most similar chunks from FAISS."""
    query_vec = embedder.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(query_vec, top_k)
    return [all_chunks[i] for i in indices[0]]  # return chunk dicts


def build_prompt(query, retrieved_chunks):
    """Assemble a clean prompt from retrieved context passages."""
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        # Label each passage so the model can reference it
        context_parts.append(
            f"[Passage {i} — {chunk['file_name']}, Page {chunk['page_number']}]\n{chunk['text']}"
        )
    context = "\n\n".join(context_parts)

    # Minimal system instruction — let the model handle language and format naturally
    system_msg = (
        "You are a helpful assistant. Answer the user's question using ONLY the "
        "provided context passages. If the answer is not in the context, say you don't know."
    )

    # Format as a chat conversation (Qwen chat template)
    messages = [
        {"role": "system",    "content": system_msg},
        {"role": "user",      "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    return messages


def format_citations(retrieved_chunks):
    """Build a deduplicated citation string from retrieved chunks."""
    seen   = set()
    citations = []
    for chunk in retrieved_chunks:
        key = (chunk["file_name"], chunk["page_number"])
        if key not in seen:
            seen.add(key)
            citations.append(f"{chunk['file_name']}, Page {chunk['page_number']}")
    return "Sumber: " + " | ".join(citations)


def ask(query):
    """Full RAG pipeline: retrieve → prompt → generate → return answer + citation."""
    # 1. Retrieve relevant chunks
    chunks   = retrieve(query)

    # 2. Build prompt and apply the model's chat template
    messages = build_prompt(query, chunks)
    prompt   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # 3. Generate answer with the LLM
    output   = llm(prompt)[0]["generated_text"]

    # 4. Strip the prompt prefix — keep only the new generated text
    answer   = output[len(prompt):].strip()

    # 5. Append citation string
    citation = format_citations(chunks)

    return answer, citation

## Minimum Output: Q&A with Citations

Run the cell below to test the chatbot with a set of example questions.

In [ ]:
sample_questions = [
    "What problem does Bitcoin solve?",
    "How does Ethereum handle smart contracts?",
    "What makes Solana faster than other blockchains?",
    "Apa itu proof of work?",
]

for question in sample_questions:
    print("=" * 70)
    print(f"Q: {question}")
    answer, citation = ask(question)
    print(f"A: {answer}")
    print(f"\n{citation}")
    print()

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: What problem does Bitcoin solve?


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Bitcoin solves the problem of maintaining a value without any backing, intrinsic value, or central issuer.

Sumber: etherium (1).pdf, Page 1 | etherium (1).pdf, Page 4 | etherium (1).pdf, Page 34

Q: How does Ethereum handle smart contracts?


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: According to the passage, Ethereum handles smart contracts through a system called "on-chain transactions". Specifically, it allows developers to create "command-line applications" that can interact with the blockchain and execute their own contracts. These contracts can then modify the global state of the network, making them similar to traditional decentralized applications (DApps) that use command-line interfaces.

Sumber: etherium (1).pdf, Page 1 | etherium (1).pdf, Page 34

Q: What makes Solana faster than other blockchains?


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: According to the passage, Solana is said to be "very likely to lead to one mining pool having a large enough percentage of the network hashpower to have de facto control over the mining process."

Sumber: solana (1).pdf, Page 1 | etherium (1).pdf, Page 26 | bitcoin (1).pdf, Page 3 | solana (1).pdf, Page 30

Q: Apa itu proof of work?
A: Proses Proof of Work adalah proses yang digunakan untuk memastikan bahwa kode blockchain tersebut tidak diubah tanpa perlu melakukan penyesuaian pada kode blockchain tersebut. Proses ini melibatkan pengurangan nilai dari sebuah nonce (nomor) dalam setiap blok kode blockchain yang diterima dan kemudian menghasilkan hash baru yang berisi zero bit. Proses ini dilakukan secara eksplisit oleh Bitcoin dengan algoritma Hashcash.

Sumber: bitcoin (1).pdf, Page 3 | solana (1).pdf, Page 19 | solana (1).pdf, Page 25 | solana (1).pdf, Page 26



## Interactive Chat Loop

Type your own question below and run the cell. Type `exit` to stop.

In [ ]:
print("RAG Chatbot ready. Type 'exit' to quit.\n")

while True:
    user_input = input("You: ").strip()
    if not user_input or user_input.lower() == "exit":
        print("Session ended.")
        break

    answer, citation = ask(user_input)
    print(f"\nBot: {answer}")
    print(f"{citation}\n")

RAG Chatbot ready. Type 'exit' to quit.

You: Jelaskan pada saya konsep transaksi bitcoin


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: Transaksi Bitcoin adalah transaksi yang dilakukan melalui sistem pembayaran berbasis teknologi digital seperti Bitcoin. Dalam Bitcoin, transaksi ini tidak memerlukan pengembalian atau penarikan uang dari satu akun ke akun lainnya. Meskipun Bitcoin memiliki tingkat volatilitas yang tinggi dan sangat cepat, transaksi Bitcoin dapat diakses oleh semua orang dengan mudah karena mereka tidak memerlukan identitas atau alamat kredit. Transaksi Bitcoin biasanya disimpan dalam sebuah daftar transaksi yang terorganisir dan dapat diakses secara otomatis oleh semua pengguna.
Sumber: etherium (1).pdf, Page 1 | etherium (1).pdf, Page 2 | solana (1).pdf, Page 1

You: jelaskan pada saya teknis transaksi bitcoin dari satu user ke user lain


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: Based on the information provided:

1. Bitcoin transactions involve two parties, each sending a certain amount of Bitcoin.

2. These transactions are recorded in a shared ledger called a blockchain.

3. The blockchain ensures that all participants have access to the same copy of the ledger.

4. Each transaction includes a timestamp indicating when it occurred.

5. The blockchain allows for multiple users to verify the authenticity of each other's transactions.

6. This makes Bitcoin transactions highly secure and tamper-proof.

7. The use of blockchain technology enables Bitcoin transactions to occur quickly and efficiently.

8. By reducing the need for intermediaries like banks, Bitcoin eliminates the risk of fraud and double spending.

9. The decentralized nature of the blockchain means that no single entity controls the network, increasing security.

10. The ability to track and verify transactions across different locations enhances privacy and reduces the likelihood of fraud